---
# Part 1: VideoMAE Training
---

## 1. Check GPU

In [ ]:
# Verify GPU is available
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Install Dependencies

In [ ]:
!pip install -q torch torchvision tqdm transformers pillow pyyaml

## 4. Clone Repository

In [ ]:
import os

REPO_PATH = '/content/Ego4d-LiteSTA'

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/saeedzns/Ego4d-LiteSTA.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git pull
    
print(f"Repository ready at: {REPO_PATH}")

## 5. Configure Paths

**⚠️ EDIT THESE PATHS TO MATCH YOUR DRIVE STRUCTURE**

In [ ]:
import os
import sys

# === EDIT THESE PATHS TO MATCH YOUR DRIVE ===
DRIVE_ROOT = '/content/drive/MyDrive'

# VideoMAE checkpoint (only needed for VideoMAE training)
VIDEOMAE_CKPT = f'{DRIVE_ROOT}/Ego4d_STA/videomae/checkpoints/videomae_ego_scratch_last.pt'

# Extracted frames directory
FRAMES_ROOT = f'{DRIVE_ROOT}/ego4d_data/v2/extracted_frames'

# Track A manifests directory
MANIFESTS_ROOT = f'{DRIVE_ROOT}/Ego4d_STA/runs/Track_A'

# Verify paths
print("Checking paths...")
print(f"{'✓' if os.path.exists(VIDEOMAE_CKPT) else '✗'} VideoMAE checkpoint: {VIDEOMAE_CKPT}")
print(f"{'✓' if os.path.exists(FRAMES_ROOT) else '✗'} Frames root: {FRAMES_ROOT}")
print(f"{'✓' if os.path.exists(MANIFESTS_ROOT) else '✗'} Manifests root: {MANIFESTS_ROOT}")

## 6. Setup Local Directory Structure

In [ ]:
import shutil
import glob

LOCAL_EXT = f'{REPO_PATH}/local_extraction'

# Create directories
os.makedirs(f'{LOCAL_EXT}/videomae_local_part/checkpoints_ego_scratch', exist_ok=True)
os.makedirs(f'{LOCAL_EXT}/v2', exist_ok=True)
os.makedirs(f'{LOCAL_EXT}/runs/Track_A', exist_ok=True)

# 1. Copy VideoMAE checkpoint
local_ckpt = f'{LOCAL_EXT}/videomae_local_part/checkpoints_ego_scratch/videomae_ego_scratch_last.pt'
if os.path.exists(VIDEOMAE_CKPT) and not os.path.exists(local_ckpt):
    print("Copying VideoMAE checkpoint...")
    shutil.copy(VIDEOMAE_CKPT, local_ckpt)
    print(f"✓ Copied to {local_ckpt}")
elif os.path.exists(local_ckpt):
    print(f"✓ VideoMAE checkpoint already exists")
else:
    print(f"⚠ VideoMAE checkpoint not found (needed for VideoMAE training only)")

# 2. Symlink frames (don't copy - too large)
local_frames = f'{LOCAL_EXT}/v2/extracted_frames'
if not os.path.exists(local_frames):
    os.symlink(FRAMES_ROOT, local_frames)
    print(f"✓ Linked frames: {local_frames}")
else:
    print(f"✓ Frames link already exists")

# 3. Copy manifests (find latest stageB run)
stageb_runs = sorted(glob.glob(f'{MANIFESTS_ROOT}/trackA_stageB_*'))
if stageb_runs:
    latest_run = stageb_runs[-1]
    run_name = os.path.basename(latest_run)
    local_run = f'{LOCAL_EXT}/runs/Track_A/{run_name}'
    
    if not os.path.exists(local_run):
        shutil.copytree(latest_run, local_run)
        print(f"✓ Copied manifests: {local_run}")
    else:
        print(f"✓ Manifests already exist: {local_run}")
    
    # Verify manifest files
    train_manifest = f'{local_run}/head_train.jsonl'
    val_manifest = f'{local_run}/head_val.jsonl'
    print(f"{'✓' if os.path.exists(train_manifest) else '✗'} Train manifest exists")
    print(f"{'✓' if os.path.exists(val_manifest) else '✗'} Val manifest exists")
else:
    print("❌ No Track A Stage B runs found!")
    print(f"   Please upload manifests to: {MANIFESTS_ROOT}/trackA_stageB_YYYYMMDD_HHMMSS/")

## 7. Set Working Directory

In [ ]:
os.chdir(LOCAL_EXT)
sys.path.insert(0, LOCAL_EXT)
print(f"Working directory: {os.getcwd()}")

## 8. Test VideoMAE Config Loading

In [ ]:
from core import load_config

cfg = load_config('trackB_videomae_ego', colab=True)

print("=== VideoMAE Config ===")
print(f"video_backbone: {cfg.get('model.tokenizer.video_backbone')}")
print(f"time_len: {cfg.get('model.tokenizer.time_len')}")
print(f"epochs: {cfg.get('training.epochs')}")
print(f"batch_size: {cfg.get('training.batch_size')}")
print(f"lr: {cfg.get('training.lr')}")
print(f"projector_in_dim: {cfg.get('model.projector.in_dim')}")

## 9. Test VideoMAE Checkpoint Loading

In [ ]:
import torch

if os.path.exists(local_ckpt):
    ckpt = torch.load(local_ckpt, map_location='cpu')
    print(f"Checkpoint keys: {list(ckpt.keys())}")
    print(f"Epoch: {ckpt.get('epoch', 'N/A')}")
    
    if 'config' in ckpt:
        print(f"Hidden size: {ckpt['config'].get('hidden_size')}")
        print(f"Num frames: {ckpt['config'].get('num_frames')}")
        print(f"Patch size: {ckpt['config'].get('patch_size')}")
else:
    print("⚠ VideoMAE checkpoint not found - skip to ResNet18 training")

## 10. Run VideoMAE Demo (Quick Verification)

In [ ]:
# Run 10-step demo to verify everything works
!python -m trackB.trackB_train_loader --config trackB_videomae_ego --demo

## 11. Full VideoMAE Training

**Expected time:** ~4-5 hours for 20 epochs on T4 GPU

In [ ]:
# Full training with VideoMAE backbone
# Adjust epochs as needed

!python -m trackB.trackB_train_loader \
    --config trackB_videomae_ego \
    --epochs 20 \
    --batch_size 8

In [ ]:
# Alternative: Quick test with fewer epochs
# !python -m trackB.trackB_train_loader --config trackB_videomae_ego --epochs 3 --batch_size 8

## 12. Save VideoMAE Results to Drive

In [ ]:
from datetime import datetime

# Create timestamped results folder
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dest = f'{DRIVE_ROOT}/Ego4d_STA/runs/Track_B_VideoMAE_{timestamp}'
os.makedirs(results_dest, exist_ok=True)

# Copy checkpoints
ckpt_src = f'{LOCAL_EXT}/runs/Track_B/checkpoints'
if os.path.exists(ckpt_src):
    shutil.copytree(ckpt_src, f'{results_dest}/checkpoints', dirs_exist_ok=True)
    print(f"✓ Saved checkpoints to {results_dest}/checkpoints")

# Copy run logs
for run_dir in glob.glob(f'{LOCAL_EXT}/runs/Track_B/trackB_*'):
    run_name = os.path.basename(run_dir)
    dest_run = f'{results_dest}/{run_name}'
    if not os.path.exists(dest_run):
        shutil.copytree(run_dir, dest_run)
        print(f"✓ Saved run: {run_name}")

print(f"\n✓ VideoMAE results saved to: {results_dest}")

---
# Part 2: ResNet18 Baseline Training
---

ResNet18 is simpler and faster - no VideoMAE checkpoint needed.

## 13. Clear Previous Run (Optional)

Run this if you want to start fresh for ResNet18 comparison

In [ ]:
# Optional: Clear previous Track B runs for clean comparison
# Uncomment to execute

# import shutil
# trackb_runs = f'{LOCAL_EXT}/runs/Track_B'
# if os.path.exists(trackb_runs):
#     shutil.rmtree(trackb_runs)
#     os.makedirs(trackb_runs)
#     print("✓ Cleared Track B runs folder")

## 14. Test ResNet18 Config Loading

In [ ]:
from core import load_config

cfg = load_config('trackB', colab=True)

print("=== ResNet18 Config ===")
print(f"video_backbone: {cfg.get('model.tokenizer.video_backbone')}")  # Should be 'resnet18'
print(f"time_len: {cfg.get('model.tokenizer.time_len')}")              # Should be 8
print(f"epochs: {cfg.get('training.epochs')}")
print(f"batch_size: {cfg.get('training.batch_size')}")
print(f"lr: {cfg.get('training.lr')}")
print(f"projector_in_dim: {cfg.get('model.projector.in_dim')}")        # Should be 512

## 15. Run ResNet18 Demo (Quick Verification)

In [ ]:
# Run 10-step demo to verify everything works
!python -m trackB.trackB_train_loader --config trackB --demo

## 16. Full ResNet18 Training

**Expected time:** ~1-2 hours for 20 epochs on T4 GPU (5x faster than VideoMAE)

In [ ]:
# Full training with ResNet18 backbone
# ResNet18 is MUCH faster than VideoMAE

!python -m trackB.trackB_train_loader \
    --config trackB \
    --epochs 20 \
    --batch_size 8

In [ ]:
# Alternative: Quick test with fewer epochs
# !python -m trackB.trackB_train_loader --config trackB --epochs 3 --batch_size 8

## 17. Save ResNet18 Results to Drive

In [ ]:
from datetime import datetime

# Create timestamped results folder
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dest = f'{DRIVE_ROOT}/Ego4d_STA/runs/Track_B_ResNet18_{timestamp}'
os.makedirs(results_dest, exist_ok=True)

# Copy checkpoints
ckpt_src = f'{LOCAL_EXT}/runs/Track_B/checkpoints'
if os.path.exists(ckpt_src):
    shutil.copytree(ckpt_src, f'{results_dest}/checkpoints', dirs_exist_ok=True)
    print(f"✓ Saved checkpoints to {results_dest}/checkpoints")

# Copy run logs
for run_dir in glob.glob(f'{LOCAL_EXT}/runs/Track_B/trackB_*'):
    run_name = os.path.basename(run_dir)
    dest_run = f'{results_dest}/{run_name}'
    if not os.path.exists(dest_run):
        shutil.copytree(run_dir, dest_run)
        print(f"✓ Saved run: {run_name}")

print(f"\n✓ ResNet18 results saved to: {results_dest}")

---
# Summary: Model Comparison
---

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                    Model Comparison Summary                       ║
╠══════════════════════════════════════════════════════════════════╣
║ Aspect          │ ResNet18              │ VideoMAE               ║
╠══════════════════════════════════════════════════════════════════╣
║ Backbone type   │ 2D CNN (per-frame)    │ 3D Transformer (video) ║
║ Token dims      │ (49, 512) = 7×7 grid  │ (1568, 768) = 8×14×14  ║
║ Frames used     │ 8 frames              │ 16 frames              ║
║ Pretrained on   │ ImageNet (web images) │ Ego4D (egocentric)     ║
║ Training speed  │ ~5× faster            │ Slower, richer features║
║ Dependencies    │ torchvision only      │ + transformers         ║
╚══════════════════════════════════════════════════════════════════╝
""")

print("\n📁 Results saved to Google Drive:")
print(f"   - VideoMAE: {DRIVE_ROOT}/Ego4d_STA/runs/Track_B_VideoMAE_*/")
print(f"   - ResNet18: {DRIVE_ROOT}/Ego4d_STA/runs/Track_B_ResNet18_*/")

---
# Troubleshooting
---

In [ ]:
# Troubleshooting: Check current directory
print(f"Current directory: {os.getcwd()}")
print(f"\nExpected: {LOCAL_EXT}")

# Fix if needed
if os.getcwd() != LOCAL_EXT:
    os.chdir(LOCAL_EXT)
    print(f"\n✓ Changed to: {os.getcwd()}")

In [ ]:
# Troubleshooting: Verify frames are accessible
local_frames = f'{LOCAL_EXT}/v2/extracted_frames'
if os.path.exists(local_frames):
    uids = os.listdir(local_frames)[:5]
    print(f"✓ Found {len(os.listdir(local_frames))} UIDs")
    print(f"  Sample UIDs: {uids}")
else:
    print(f"✗ Frames not found at {local_frames}")

In [ ]:
# Troubleshooting: Check manifests
import glob
manifests = glob.glob(f'{LOCAL_EXT}/runs/Track_A/*/head_train.jsonl')
if manifests:
    print(f"✓ Found {len(manifests)} manifest(s)")
    for m in manifests:
        with open(m, 'r') as f:
            lines = len(f.readlines())
        print(f"  {m}: {lines} records")
else:
    print("✗ No manifests found")

In [ ]:
# Troubleshooting: CUDA out of memory - reduce batch size
# !python -m trackB.trackB_train_loader --config trackB_videomae_ego --batch_size 4
# !python -m trackB.trackB_train_loader --config trackB --batch_size 4